In [1]:
%load_ext autoreload
%autoreload 2

from explain.llm import create_client

## Test base conversation with client

### Claude


In [3]:
client = create_client(provider="anthropic", model="claude-sonnet-4@20250514")

resp = client.generate([{"role": "user", "content": "Hello, how are you?"}])

2025-08-04 16:52:42.384 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: anthropic


### OpenAI

In [4]:
client = create_client(provider="openai", model="gpt-4.1")

client.generate([{"role": "user", "content": "Hello, how are you?"}])

2025-08-04 16:52:51.628 | INFO     | explain.llm._client:__init__:472 - Initialized OpenAI client with model gpt-4.1
2025-08-04 16:52:51.629 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: openai


LLMResponse(content="Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", tool_calls=None, messages=[{'role': 'user', 'content': 'Hello, how are you?'}, {'role': 'assistant', 'content': "Hello! I'm just a program, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?"}])

### Gemini

In [5]:
client = create_client(provider="gemini", model="gemini-2.5-flash")

client.generate([{"role": "user", "content": "Hello, how are you?"}])

2025-08-04 16:52:53.238 | INFO     | explain.llm._client:__init__:280 - Initialized Google Gemini client with model gemini-2.5-flash
2025-08-04 16:52:53.238 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: gemini


LLMResponse(content="Hello! I'm functioning perfectly, thank you for asking.\n\nAs an AI, I don't have feelings or a physical state, but I'm ready and eager to assist you.\n\nHow can I help you today?", tool_calls=None, messages=[{'role': 'user', 'content': 'Hello, how are you?'}, {'role': 'assistant', 'content': "Hello! I'm functioning perfectly, thank you for asking.\n\nAs an AI, I don't have feelings or a physical state, but I'm ready and eager to assist you.\n\nHow can I help you today?"}])

### LiteLLM


In [9]:
client = create_client(provider="litellm", model="gemini-2.5-flash")
client.generate([{"role": "user", "content": "Hello, how are you?"}])

2025-08-04 16:53:55.482 | INFO     | explain.llm._client:__init__:577 - Initialized LiteLLM client with model gemini-2.5-flash
2025-08-04 16:53:55.483 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: litellm


LLMResponse(content="Hello!\n\nAs an AI, I don't experience feelings or have a physical state, but I'm functioning perfectly and ready to assist you.\n\nHow can I help you today?", tool_calls=None, messages=[{'role': 'user', 'content': 'Hello, how are you?'}, {'role': 'assistant', 'content': "Hello!\n\nAs an AI, I don't experience feelings or have a physical state, but I'm functioning perfectly and ready to assist you.\n\nHow can I help you today?"}])

## Test tool

In [13]:
import json

from explain.eval.tools import BIOLOGICAL_TOOLS, REGISTERED_TOOLS, ToolVerifier
from explain.llm import create_client

prompt = "Can you check the drug-target interaction for the drug 'imatinib' and the target 'BCR-ABL'?"
providers = ["litellm", "openai", "gemini", "anthropic"]  # Add "openai", "gemini" as needed

for provider in providers:
    print(f"--- Testing Tool Usage for {provider.upper()} ---")
    try:
        client = create_client(provider=provider)
        messages = [{"role": "user", "content": prompt}]
        print(f"User > {prompt}")

        # First model call
        response = client.generate(messages=messages, tools=list(BIOLOGICAL_TOOLS.values()))
        messages = response.messages  # Updated history, already formatted per provider

        # Execute tools
        if response.tool_calls:
            tool_outputs = []
            for tool_call in response.tool_calls:
                tool_name = tool_call["function"]["name"]
                tool_args = tool_call["function"]["arguments"]
                if tool_name in REGISTERED_TOOLS:
                    feedback = ToolVerifier.call_tool(tool_call["function"])
                    tool_outputs.append(
                        {
                            "tool_call_id": tool_call["id"],
                            "name": tool_name,
                            "content": feedback,
                        }
                    )
                    print(f"Tool output > {json.dumps(feedback, indent=2)}")

            # Add tool results to messages
            tool_response_messages = client.format_tool_response(tool_outputs)
            messages.extend(tool_response_messages)

            # Second model call with tool results
            final_response = client.generate(messages=messages, tools=list(BIOLOGICAL_TOOLS.values()))
            print(f"Assistant > {final_response.content}")

    except Exception as e:
        import traceback

        print(f"An error occurred while testing {provider}: {e}")
        traceback.print_exc()

    print("\n" + "=" * 60 + "\n")

2025-08-04 16:55:09.196 | INFO     | explain.llm._client:__init__:577 - Initialized LiteLLM client with model claude-sonnet-4@20250514
2025-08-04 16:55:09.197 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: litellm


--- Testing Tool Usage for LITELLM ---
User > Can you check the drug-target interaction for the drug 'imatinib' and the target 'BCR-ABL'?
Tool output > "{\n  \"reward\": 0.0,\n  \"feedback\": {\n    \"drug\": \"imatinib\",\n    \"target\": \"BCR-ABL\",\n    \"interaction_type\": null,\n    \"strength_uM\": null,\n    \"verification_status\": \"NOT_VERIFIED\"\n  }\n}"
Assistant > The drug-target interaction check for imatinib and BCR-ABL shows:

- **Drug**: imatinib
- **Target**: BCR-ABL
- **Interaction Type**: Not specified in the database
- **Strength**: Not specified (no concentration value available)
- **Verification Status**: NOT_VERIFIED

This indicates that while the system recognizes both imatinib and BCR-ABL, the specific interaction details (type and strength) are not verified or available in the current database. In clinical practice, imatinib is well-known as a BCR-ABL tyrosine kinase inhibitor used to treat chronic myeloid leukemia (CML), but this particular database query 

2025-08-04 16:55:19.506 | INFO     | explain.llm._client:__init__:472 - Initialized OpenAI client with model gpt-4.1
2025-08-04 16:55:19.507 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: openai


User > Can you check the drug-target interaction for the drug 'imatinib' and the target 'BCR-ABL'?
Tool output > "{\n  \"reward\": 0.0,\n  \"feedback\": {\n    \"drug\": \"imatinib\",\n    \"target\": \"BCR-ABL\",\n    \"interaction_type\": null,\n    \"strength_uM\": null,\n    \"verification_status\": \"NOT_VERIFIED\"\n  }\n}"


2025-08-04 16:55:25.492 | INFO     | explain.llm._client:__init__:280 - Initialized Google Gemini client with model gemini-2.5-flash
2025-08-04 16:55:25.493 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: gemini


Assistant > I could not verify a specific drug-target interaction between imatinib and BCR-ABL in the current database. However, imatinib is widely known in scientific literature as a tyrosine kinase inhibitor that specifically targets the BCR-ABL fusion protein, which is characteristic of chronic myeloid leukemia (CML).

If you need more detailed or specific interaction data (such as interaction type or binding strength), please let me know!


--- Testing Tool Usage for GEMINI ---
User > Can you check the drug-target interaction for the drug 'imatinib' and the target 'BCR-ABL'?


Tool output > "{\n  \"reward\": 1.0,\n  \"feedback\": {\n    \"drug\": \"imatinib\",\n    \"target\": \"BCR-ABL\",\n    \"interaction_type\": null,\n    \"strength_uM\": null,\n    \"verification_status\": \"VERIFIED\"\n  }\n}"


2025-08-04 16:55:27.569 | INFO     | explain.llm._client:__init__:694 - Initialized unified LLM client with provider: anthropic


Assistant > I found that imatinib interacts with BCR-ABL. 



--- Testing Tool Usage for ANTHROPIC ---
User > Can you check the drug-target interaction for the drug 'imatinib' and the target 'BCR-ABL'?
Tool output > "{\n  \"reward\": 1.0,\n  \"feedback\": {\n    \"drug\": \"imatinib\",\n    \"target\": \"BCR-ABL\",\n    \"interaction_type\": null,\n    \"strength_uM\": null,\n    \"verification_status\": \"VERIFIED\"\n  }\n}"
Assistant > The drug-target interaction check confirms that **imatinib does interact with BCR-ABL**. The interaction has been verified with a high confidence score (reward: 1.0).

This is a well-established interaction - imatinib (Gleevec) is a tyrosine kinase inhibitor that specifically targets the BCR-ABL fusion protein, which is the hallmark of chronic myeloid leukemia (CML). This drug-target interaction is the basis for imatinib's therapeutic efficacy in treating CML and certain other cancers with BCR-ABL mutations.




In [17]:
import pandas as pd
df = pd.read_csv("../../data/curation_v1/results/structure-explain-results-v1.csv")

In [20]:
print(df.iloc[0].dag)

edge("n1", "n2", relation="causal")
edge("n2", "n3", relation="causal")
edge("n2", "n4", relation="causal")
edge("n3", "n5", relation="correlative")
edge("n4", "n6", relation="causal")
edge("n3", "n7", relation="causal")
edge("n2", "n8", relation="causal")


In [21]:
print(df.iloc[0].explain)

set_context(id="n0", cell_type="endothelial cell", disease_context="tumor angiogenesis", prior_perturbation="VEGF addition")
binds_to(id="n1", actor="Bevacizumab", target="VEGF-A", affinity="0.15 μg/ml", via="monoclonal antibody recognition")
modulates_activity(id="n2", entity="VEGFR-2", direction="down", via="ligand sequestration preventing receptor binding")
modulates_activity(id="n3", entity="PI3K/Akt pathway", direction="down", via="blocked VEGFR-2 autophosphorylation")
modulates_activity(id="n4", entity="MAPK/ERK pathway", direction="down", via="blocked VEGFR-2 autophosphorylation")
regulates_expression(id="n5", source="PI3K/Akt pathway", gene_or_signature=["hypoxia-responsive genes", "cell cycle genes"], direction="down", via="reduced transcriptional activation")
causes_phenotype(id="n6", source="MAPK/ERK pathway", phenotype="decreased endothelial cell proliferation", via="impaired growth signaling")
causes_phenotype(id="n7", source="PI3K/Akt pathway", phenotype="increased endoth